# Adult Income

## Câu 6. Tách tập huấn luyện và kiểm thử

### Yêu cầu

Tách dữ liệu thành các phần huấn luyện và kiểm thử theo ranh giới đã xác định.

### Dữ liệu và phương pháp

Thuộc tính quyết định: income. Sử dụng toàn bộ dữ liệu được cung cấp. Giữ nguyên nguồn train/test sau khi loại dòng trống; nối hai tập để thực hiện quy trình của đề. Tương quan cao được quy ước là |r| ≥ 0,8 giữa các thuộc tính đầu vào; giữ cột đứng trước trong thứ tự gốc.

In [1]:
print("Đặng Ngọc Hiếu - 6451071024")

from pathlib import Path
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "datasets").is_dir():
    if (ROOT / "6451071024_Lab2" / "datasets").is_dir():
        ROOT = ROOT / "6451071024_Lab2"
        break
    if ROOT == ROOT.parent:
        raise FileNotFoundError("Không tìm thấy thư mục datasets của LAB2.")
    ROOT = ROOT.parent
pd.set_option("display.max_columns", None)


train_raw = pd.read_csv(ROOT / "datasets/adult/adult.data.csv", skipinitialspace=True)
test_raw = pd.read_csv(ROOT / "datasets/adult/adult.test.csv", skipinitialspace=True)
if "income" not in train_raw or "income" not in test_raw:
    raise ValueError("Hai tệp Adult phải có cột income.")
if "fnlwgt" not in train_raw or "fnlwgt" in test_raw:
    raise ValueError("Kiểm tra hai phiên bản Adult: fnlwgt chỉ có trong tập huấn luyện.")


Đặng Ngọc Hiếu - 6451071024


In [2]:
print("Đặng Ngọc Hiếu - 6451071024")

# Kiểu string tường minh giúp nhận diện thiếu mà không dựa vào downcasting.
def clean_adult(frame):
    frame = frame.copy()
    for column in frame.select_dtypes(include=["object", "string"]).columns:
        values = frame[column].astype("string").str.strip()
        frame[column] = values.mask(values.isin(["?", ""]))
    return frame.dropna()

train = clean_adult(train_raw).drop(columns="fnlwgt")
test = clean_adult(test_raw)
if list(train.columns) != list(test.columns):
    raise ValueError("Tên và thứ tự cột Adult chưa khớp sau khi bỏ fnlwgt.")
n_train = len(train)
data = pd.concat([train, test], ignore_index=True)
TARGET = "income"


Đặng Ngọc Hiếu - 6451071024


In [3]:
print("Đặng Ngọc Hiếu - 6451071024")

# Tái hiện bước chọn thuộc tính của câu 3, không vẽ lại heatmap.
corr = data.drop(columns=TARGET).select_dtypes(include="number").corr(method="pearson")
CORRELATION_THRESHOLD = 0.8
# |r| >= 0,8 là quy ước đã nêu trong phương pháp, không phải ngưỡng đề ấn định.
drop_columns = []
for j, column in enumerate(corr.columns):
    for previous in corr.columns[:j]:
        value = corr.loc[previous, column]
        if previous not in drop_columns and pd.notna(value) and abs(value) >= CORRELATION_THRESHOLD:
            drop_columns.append(column)
            break
data = data.drop(columns=drop_columns)


Đặng Ngọc Hiếu - 6451071024


In [4]:
print("Đặng Ngọc Hiếu - 6451071024")

features = data.drop(columns=TARGET)
labels = data[TARGET]


Đặng Ngọc Hiếu - 6451071024


In [5]:
print("Đặng Ngọc Hiếu - 6451071024")

non_numeric = features.select_dtypes(exclude=["number", "bool"]).columns.tolist()
# uint8 lưu đúng 0/1 với một byte cho mỗi phần tử one-hot.
features_onehot = pd.get_dummies(features, columns=non_numeric, dtype="uint8")


Đặng Ngọc Hiếu - 6451071024


In [6]:
print("Đặng Ngọc Hiếu - 6451071024")

# Giữ đúng ranh giới của hai tệp Adult sau làm sạch.
X_train = features_onehot.iloc[:n_train]
X_test = features_onehot.iloc[n_train:]
y_train = labels.iloc[:n_train]
y_test = labels.iloc[n_train:]

if X_train.empty or X_test.empty:
    raise ValueError("Tập huấn luyện và kiểm thử không được trống.")

display(pd.DataFrame({"Tập": ["Huấn luyện", "Kiểm thử"],
    "Số mẫu": [len(X_train), len(X_test)],
    "Số thuộc tính": [X_train.shape[1], X_test.shape[1]]}))


Đặng Ngọc Hiếu - 6451071024


,Tập,Số mẫu,Số thuộc tính
0,Huấn luyện,30162,103
1,Kiểm thử,15060,103
